In [1]:
# Cell 1: Import libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import autokeras as ak
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from ast import literal_eval
import matplotlib.pyplot as plt
import os


2025-04-08 20:27:51.379459: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-08 20:27:51.394458: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744169271.408281   25140 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744169271.412738   25140 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-08 20:27:51.428369: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
# Cell 2: Load and prepare data
# Adjust path for WSL2
file_path = '~/cerv_spine.csv'
df = pd.read_csv(file_path)

# Print basic dataset info
print(f"Dataset shape: {df.shape}")
print(f"Class distribution: \n{df['patient_overall'].value_counts()}")

# Convert embedding strings to numpy arrays
df['embedding'] = df['embedding'].apply(literal_eval)

# Extract features and target
X = np.array(df['embedding'].tolist())
y = df['patient_overall'].values

print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")


Dataset shape: (559, 11)
Class distribution: 
patient_overall
0    284
1    275
Name: count, dtype: int64
Feature shape: (559, 1408)
Target shape: (559,)


In [3]:
# Cell 3: Feature preprocessing
# Normalize embeddings using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data with stratification to handle imbalanced classes
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# Further split training data to create validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_train
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")


Training set: (357, 1408)
Validation set: (90, 1408)
Test set: (112, 1408)


In [6]:
# Cell 5: Define the custom AutoKeras model
# Create the input node for 1408-d feature vectors.
input_node = ak.Input(shape=(1408,))

# Create two DenseBlock layers letting AutoKeras search over number of layers, units, and dropout
x = ak.DenseBlock(use_batchnorm=True)(input_node)
x = ak.DenseBlock(use_batchnorm=True)(x)

# Define the classification head; setting dropout here is optional.
output_node = ak.ClassificationHead(dropout=0.25, num_classes=2)(x)

# Option A (Recommended): Do not supply a custom tuner.
model = ak.AutoModel(
    inputs=input_node,
    outputs=output_node,
    max_trials=50,
    overwrite=True,
    project_name='spine_classifier_improved_corrected'
)

# Option B: If you want to use a custom tuner such as Hyperband, uncomment the following lines.
# Note: This workaround sets the tuner’s hypermodel manually to avoid the NoneType error.
# custom_tuner = ak.Hyperband(
#     factor=3,
#     objective='val_accuracy',
#     max_epochs=200,
#     directory='tuner_dir_corrected',
#     project_name='spine_tuning_corrected'
# )
# model = ak.AutoModel(
#     inputs=input_node,
#     outputs=output_node,
#     max_trials=50,
#     overwrite=True,
#     project_name='spine_classifier_improved_corrected',
#     tuner=custom_tuner
# )
# # Manually assign the hypermodel to the tuner to avoid the error.
# model.tuner.hypermodel = model.hypermodel

print("AutoModel defined successfully.")


AutoModel defined successfully.


In [7]:
# Cell 6: Define callbacks for training
# Several callbacks are set to improve training (early stopping, reducing learning rate, TensorBoard logging, and model checkpointing)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1
)
reduce_lr_loss = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)
tensorboard_cb = tf.keras.callbacks.TensorBoard(log_dir='./logs', histogram_freq=1)
mcp_save = tf.keras.callbacks.ModelCheckpoint(
    'best_model.h5', save_best_only=True, monitor='val_loss', mode='min'
)
callbacks = [early_stopping, reduce_lr_loss, tensorboard_cb, mcp_save]


In [8]:
# Cell 7: Train the model
# Train with extended epochs and use the validation set defined earlier.
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)


Trial 50 Complete [00h 00m 17s]
val_loss: 0.6927875280380249

Best val_loss So Far: 0.6545105576515198
Total elapsed time: 00h 12m 01s


/home/bahram/.local/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 38 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [9]:
# Cell 8: Evaluate the best model and generate predictions
best_model = model.export_model()  # Export the best Keras model.
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Generate predictions (handle binary output appropriately)
y_pred = best_model.predict(X_test)
if y_pred.shape[1] > 1:
    y_pred_classes = np.argmax(y_pred, axis=1)
else:
    y_pred_classes = (y_pred > 0.5).astype("int32")
    
# Display a classification report and confusion matrix.
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_classes))
auc = roc_auc_score(y_test, y_pred) if y_pred.shape[1] == 1 else roc_auc_score(tf.keras.utils.to_categorical(y_test), y_pred)
print(f"ROC-AUC Score: {auc:.4f}")


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 226ms/step - accuracy: 0.5554 - loss: 0.6855
Test Loss: 0.6872
Test Accuracy: 0.5446


/home/bahram/.local/lib/python3.10/site-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(32, 1408))
  warnings.warn(msg)


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step 

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.30      0.40        57
           1       0.52      0.80      0.63        55

    accuracy                           0.54       112
   macro avg       0.57      0.55      0.52       112
weighted avg       0.57      0.54      0.51       112


Confusion Matrix:
[[17 40]
 [11 44]]
ROC-AUC Score: 0.6450
